In [1]:
!pip -q install pyspark

In [3]:
from google.colab import files
from pyspark.sql import SparkSession

In [4]:
from google.colab import files
uploaded = files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv


In [5]:
from google.colab import files
uploaded = files.upload()

Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv


In [6]:
from google.colab import files
uploaded = files.upload()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [8]:
from pyspark.sql.functions import col

# Initialize SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Load the uploaded sales CSV file
sales_df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)

# Filter the dataframe
filtered_df = sales_df.filter(col("quantity") > 5)
filtered_df.show(5)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
+--------+----------+--------+----------+--------+----------+---

In [9]:
selected_df = sales_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
)
selected_df.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

In [10]:
expensive_df = sales_df.filter(col("unit_price") > 20000)
print("Matching records:", expensive_df.count())

Matching records: 6560


In [11]:
gross_df = sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
gross_df.explain()
gross_df.show()

== Physical Plan ==
*(1) Project [order_id#17, order_date#18, store_id#19, product_id#20, quantity#21, unit_price#22, discount_pct#23, payment_method#24, order_status#25, sales_channel#26, (quantity#21 * unit_price#22) AS gross_amount#101]
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------+----------+--------+-

In [12]:
completed_sorted_df = sales_df.filter(
    col("order_status") == "Completed"
).orderBy(col("unit_price").desc())
print(completed_sorted_df.take(10))

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'), Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'), Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'), Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'), Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, un

In [13]:
pipeline_df = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_amount"
).orderBy(
    col("gross_amount").desc()
)
pipeline_df.show()

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

In [15]:
from pyspark.sql.functions import col, round
discounted_df = sales_df.filter(
    col("discount_pct") >= 10
).withColumn(
    "discounted_sales_amount",
    round(
        col("quantity") * col("unit_price") *
        (1 - col("discount_pct") / 100),
        2
    )
).orderBy(
    col("discounted_sales_amount").desc()
)
discounted_df.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|               647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|               647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|               647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|               647991.9|
| O007743|202

In [17]:
from pyspark.sql.functions import col, sum
payment_summary_df = sales_df.groupBy(
    "payment_method"
).agg(
    sum("quantity").alias("total_quantity_sold")
).orderBy(
    col("total_quantity_sold").desc()
)
payment_summary_df.show()

+--------------+-------------------+
|payment_method|total_quantity_sold|
+--------------+-------------------+
|          Cash|              31754|
|          Card|              31327|
|           UPI|              30894|
|   Net Banking|              30889|
+--------------+-------------------+



In [19]:
from pyspark.sql.functions import count
channel_status_df = sales_df.groupBy(
    "sales_channel",
    "order_status"
).agg(
    count("*").alias("transaction_count")
)
channel_status_df.show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|       Online|   Completed|            10797|
|     In Store|   Cancelled|              592|
|       Online|    Returned|              987|
|       Online|   Cancelled|              633|
|     In Store|    Returned|             1013|
|     In Store|   Completed|            10978|
+-------------+------------+-----------------+



In [20]:
orders_df = sales_df.select(
    "order_id",
    "store_id",
    "product_id",
    "quantity"
)
orders_without_store_df = orders_df.drop("store_id")
print(orders_without_store_df.first())

Row(order_id='O000001', product_id='P002', quantity=3)


In [22]:
# Load products_df from the uploaded CSV
products_df = spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)
sales_products_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).select(
    "product_name",
    "category",
    "quantity",
    "unit_price"
)
sales_products_df.show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [24]:
# Load stores_df from the uploaded CSV
stores_df = spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)
city_sales_df = sales_df.join(
    stores_df,
    on="store_id",
    how="inner"
).groupBy(
    "city"
).agg(
    sum("quantity").alias("total_quantity_sold")
)
city_sales_df.show()

+---------+-------------------+
|     city|total_quantity_sold|
+---------+-------------------+
|  Chennai|              15727|
|   Mumbai|              15489|
|  Kolkata|              15619|
|     Pune|              15407|
|    Delhi|              15101|
|Bengaluru|              15718|
|Hyderabad|              15784|
|   Jaipur|              16019|
+---------+-------------------+



In [25]:
completed_sales_df = sales_df.filter(
    col("order_status") == "Completed"
)
all_joined_df = completed_sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).join(
    stores_df,
    on="store_id",
    how="inner"
)
category_region_revenue_df = all_joined_df.withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)
).groupBy(
    "category",
    "region"
).agg(
    round(sum("discounted_revenue"), 2).alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)
category_region_revenue_df.show()

+-----------+------+--------------+
|   category|region| total_revenue|
+-----------+------+--------------+
|  Computers| South|3.5266921295E8|
|  Computers| North| 2.273618241E8|
|  Computers|  West|2.0921530285E8|
|  Computers|  East|1.1648428645E8|
|    Mobiles| South| 1.144250127E8|
|    Mobiles|  West|    7.639817E7|
|    Mobiles| North|  7.14025309E7|
|    Tablets| South| 6.050691915E7|
|   Displays| South|  4.34817421E7|
|    Mobiles|  East|   3.8912225E7|
|    Tablets| North| 3.653276155E7|
|    Tablets|  West| 3.556450365E7|
|    Storage| South| 3.506146435E7|
|      Audio| South| 3.273420575E7|
|   Displays| North| 3.095262915E7|
|   Displays|  West|  2.67105287E7|
|Accessories| South|  2.66420428E7|
|      Audio|  West| 2.395121565E7|
|      Audio| North|  2.30163134E7|
|    Storage| North|  2.29309727E7|
+-----------+------+--------------+
only showing top 20 rows


In [26]:
product_revenue_df = sales_df.filter(
    col("order_status") == "Completed"
).join(
    products_df,
    on="product_id",
    how="inner"
).withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)
).groupBy(
    "product_id",
    "product_name"
).agg(
    round(sum("discounted_revenue"), 2).alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)
product_revenue_df.show(5)

+----------+---------------+--------------+
|product_id|   product_name| total_revenue|
+----------+---------------+--------------+
|      P007|      Laptop 16|5.2272546585E8|
|      P006|      Laptop 14| 3.830051605E8|
|      P014|  Smartphone 5G|2.0604063175E8|
|      P008| Android Tablet| 1.504065603E8|
|      P004|27-inch Monitor|1.1406074015E8|
+----------+---------------+--------------+
only showing top 5 rows


In [27]:
step1_df = sales_df.filter(
    col("order_status") == "Completed"
)
print("Transformation 1: Filter completed orders")
step2_df = step1_df.select(
    "payment_method",
    "quantity",
    "unit_price",
    "discount_pct"
)
print("Transformation 2: Select columns")
step3_df = step2_df.withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)
)
print("Transformation 3: Calculate discounted revenue")
step4_df = step3_df.groupBy(
    "payment_method"
).agg(
    sum("discounted_revenue").alias("total_revenue")
)
print("Transformation 4: Group and aggregate")
step5_df = step4_df.orderBy(
    col("total_revenue").desc()
)
print("Transformation 5: Sort results")
step5_df.show()

Transformation 1: Filter completed orders
Transformation 2: Select columns
Transformation 3: Calculate discounted revenue
Transformation 4: Group and aggregate
Transformation 5: Sort results
+--------------+--------------------+
|payment_method|       total_revenue|
+--------------+--------------------+
|   Net Banking|4.3895969179999954E8|
|           UPI| 4.379356435499997E8|
|          Cash|4.3289882239999884E8|
|          Card|4.1456176919999975E8|
+--------------+--------------------+



In [28]:
filtered_plan_df = sales_df.filter(
    col("quantity") > 5
)
filtered_plan_df.explain()
filtered_plan_df.show()
print("Total matching records:", filtered_plan_df.count())

== Physical Plan ==
*(1) Filter (isnotnull(quantity#21) AND (quantity#21 > 5))
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [isnotnull(quantity#21), (quantity#21 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,5)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      

In [29]:
category_quantity_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).groupBy(
    "category"
).agg(
    sum("quantity").alias("total_quantity")
)
category_quantity = category_quantity_df.collect()
for row in category_quantity:
    print(row)

Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Accessories', total_quantity=41839)
Row(category='Storage', total_quantity=17251)
Row(category='Audio', total_quantity=17159)
Row(category='Tablets', total_quantity=7915)


In [30]:
final_df = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "discounted_revenue",
    round(
        col("quantity") * col("unit_price") *
        (1 - col("discount_pct") / 100),
        2
    )
)
final_df.write.mode("overwrite").parquet(
    "/content/completed_sales_parquet"
)